# HM4SR Kaggle All-in-One Notebook

This notebook is designed to run on Kaggle in a mostly self-contained way.

Included inline:
- `recbole_model/HM4SR.py`
- `dataprocess/*.py`
- `config/data.yaml` and `config/Games.yaml`

External requirement:
- `recbole` package (installed automatically if missing)
- Dataset files under `./dataset/Games/`


In [ ]:
from __future__ import annotations

import importlib
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
print("ROOT =", ROOT)

def ensure_package(pkg: str):
    try:
        importlib.import_module(pkg)
        print(f"{pkg} already installed")
    except Exception:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

ensure_package("recbole")


## 1) Write inline config files


In [ ]:
CONFIG_DIR = ROOT / "config"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML = r"""
seed: 2025
show_progress: False
topk: [5, 10, 20, 50]
metrics: ["NDCG", "MRR"]
field_separator: "\t"
seq_separator: " "
USER_ID_FIELD: user_id
ITEM_ID_FIELD: item_id
# RATING_FIELD: stars
TIME_FIELD: timestamp
# NEG_PREFIX: neg_
ITEM_LIST_LENGTH_FIELD: item_length
LIST_SUFFIX: _list
MAX_ITEM_LIST_LENGTH: 50
shuffle: True
POSITION_FIELD: position_id
load_col:
    inter: [user_id, item_id, timestamp]
    item: ['item_id','title','sales_rank','price','brand','categories','sales_type']

user_inter_num_interval: "[5,inf)"
item_inter_num_interval: "[5,inf)"
# val_interval:
#   stars: "(0,inf)"

epochs: 100
train_batch_size: 1024
learner: adam
learning_rate: 0.001
eval_batch_size: 2048
valid_metric: NDCG@20
valid_metric_bigger: True
eval_args:
  split: { "LS": "valid_and_test" }
  group_by: user
  order: TO
  mode: full
neg_sampling: ~
train_neg_sample_args: ~
eval_step: 1
stopping_step: 10
loss_decimal_place: 5
metric_decimal_place: 5
training_neg_sample_num: 0
data_path: 'dataset/'
"""
GAMES_YAML = r"""
# SASRec
n_layers: 2
n_heads: 2
hidden_size: 64
inner_size: 256
hidden_dropout_prob: 0.5
attn_dropout_prob: 0.5
hidden_act: "gelu"
layer_norm_eps: 1e-12
initializer_range: 0.02
initializer_weight: [0.0, 1.0, 1.0]
loss_type: "CE"
# ID Contrast Learning
temperature: 0.2
# Placeholder Contrast Learning
beta: 0.3
phcl_temperature: 1.0
phcl_weight: 1.0
# Representation MoE
expert_num: 4
gate_selection: softmax
start_expert_num: 4
start_gate_selection: softmax
# Temporal MoE
temporal_expert_num: 4
temporal_gate_selection: softmax
interval_scale: 100
"""

(CONFIG_DIR / "data.yaml").write_text(DATA_YAML.strip() + "\n", encoding="utf-8")
(CONFIG_DIR / "Games.yaml").write_text(GAMES_YAML.strip() + "\n", encoding="utf-8")
print("Wrote:", CONFIG_DIR / "data.yaml")
print("Wrote:", CONFIG_DIR / "Games.yaml")


## 2) Inline HM4SR model code


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from recbole.model.abstract_recommender import SequentialRecommender
from recbole.model.layers import TransformerEncoder
import pickle
import math
import random


class HM4SR(SequentialRecommender):
    def __init__(self, config, dataset):
        super(HM4SR, self).__init__(config, dataset)
        self.data_name = config["dataset"].split('/')[-1]
        self.n_layers = config["n_layers"]
        self.n_heads = config["n_heads"]
        self.hidden_size = config["hidden_size"]
        self.inner_size = config["inner_size"]
        self.hidden_dropout_prob = config["hidden_dropout_prob"]
        self.attn_dropout_prob = config["attn_dropout_prob"]
        self.hidden_act = config["hidden_act"]
        self.layer_norm_eps = config["layer_norm_eps"]
        self.initializer_range = config["initializer_range"]
        self.loss_type = config["loss_type"]
        self.temperature = config["temperature"]
        self.phcl_temperature = config["phcl_temperature"]
        self.phcl_weight = config["phcl_weight"]
        self.beta = config["beta"]

        self.item_embedding = nn.Embedding(self.n_items, self.hidden_size, padding_idx=0)
        self.position_embedding = nn.Embedding(self.max_seq_length, self.hidden_size)
        self.item_seq = TransformerEncoder(
            n_layers=self.n_layers, n_heads=self.n_heads,
            hidden_size=self.hidden_size, inner_size=self.inner_size, hidden_dropout_prob=self.hidden_dropout_prob, attn_dropout_prob=self.attn_dropout_prob,
            hidden_act=self.hidden_act, layer_norm_eps=self.layer_norm_eps)
        self.txt_seq = TransformerEncoder(
            n_layers=self.n_layers, n_heads=self.n_heads,
            hidden_size=self.hidden_size, inner_size=self.inner_size, hidden_dropout_prob=self.hidden_dropout_prob,
            attn_dropout_prob=self.attn_dropout_prob,
            hidden_act=self.hidden_act, layer_norm_eps=self.layer_norm_eps)
        self.img_seq = TransformerEncoder(
            n_layers=self.n_layers, n_heads=self.n_heads,
            hidden_size=self.hidden_size, inner_size=self.inner_size, hidden_dropout_prob=self.hidden_dropout_prob,
            attn_dropout_prob=self.attn_dropout_prob,
            hidden_act=self.hidden_act, layer_norm_eps=self.layer_norm_eps)

        self.item_ln = nn.LayerNorm(self.hidden_size, eps=self.layer_norm_eps)
        self.txt_ln = nn.LayerNorm(self.hidden_size, eps=self.layer_norm_eps)
        self.img_ln = nn.LayerNorm(self.hidden_size, eps=self.layer_norm_eps)

        # 增加模态映射
        self.txt_projection = nn.Linear(768, self.hidden_size)
        self.img_projection = nn.Linear(768, self.hidden_size)

        self.dropout = nn.Dropout(self.hidden_dropout_prob)

        self.loss_fct = nn.CrossEntropyLoss()

        # 增加时序信息
        self.time_moe = Temporal_MoE_C(config)

        self.apply(self._init_weights)

        # 增加模态嵌入
        self.txt_embedding = nn.Embedding.from_pretrained(torch.load(f'./dataset/{self.data_name}/txt_emb.pt'))
        self.img_embedding = nn.Embedding.from_pretrained(torch.load(f'./dataset/{self.data_name}/img_emb.pt'))
        # 增加属性类别预测任务
        cat_emb = torch.load(f'./dataset/{self.data_name}/cat.pt').float()
        self.cat_embedding = nn.Embedding.from_pretrained(cat_emb)
        self.cat_linear = nn.Linear(3 * self.hidden_size, cat_emb.shape[-1])
        self.cat_criterion = nn.BCEWithLogitsLoss()
        # 增加初始MoE
        self.start_moe = Align_MoE(config)
        # 增加placeholder编码器
        self.placeholder_txt = nn.Linear(2*self.hidden_size, self.hidden_size)
        self.placeholder_img = nn.Linear(2*self.hidden_size, self.hidden_size)

    def _init_weights(self, module):
        """Initialize the weights"""
        if isinstance(module, (nn.Linear, nn.Embedding)):
            module.weight.data.normal_(mean=0.0, std=self.initializer_range)
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)
        if isinstance(module, nn.Linear) and module.bias is not None:
            module.bias.data.zero_()

    def forward(self, input_idx, seq_length, timestamp=None):
        # 嵌入映射
        item_emb = self.item_embedding(input_idx)
        txt_emb = self.txt_projection(self.txt_embedding(input_idx))
        img_emb = self.img_projection(self.img_embedding(input_idx))
        # 位置嵌入
        id_pos_emb = self.position_embedding.weight[:input_idx.shape[1]]
        id_pos_emb = id_pos_emb.unsqueeze(0).repeat(item_emb.shape[0], 1, 1)
        item_emb += id_pos_emb
        txt_emb += id_pos_emb
        img_emb += id_pos_emb
        ### 添加MoE ###
        align_info = self.start_moe(torch.cat([item_emb, txt_emb, img_emb], dim=-1))
        item_emb += align_info[0]
        txt_emb += align_info[1]
        img_emb += align_info[2]
        ### 添加时序MoE ###
        item_emb, txt_emb, img_emb = self.time_moe(torch.cat([item_emb, txt_emb, img_emb], dim=-1), timestamp)
        # 层正则化+dropout
        item_emb_o = self.dropout(self.item_ln(item_emb))
        txt_emb_o = self.dropout(self.txt_ln(txt_emb))
        img_emb_o = self.dropout(self.img_ln(img_emb))
        # 序列编码
        extended_attention_mask = self.get_attention_mask(input_idx)
        item_seq_full = self.item_seq(item_emb_o, extended_attention_mask, output_all_encoded_layers=True)[-1]
        txt_seq_full = self.txt_seq(txt_emb_o, extended_attention_mask, output_all_encoded_layers=True)[-1]
        img_seq_full = self.img_seq(img_emb_o, extended_attention_mask, output_all_encoded_layers=True)[-1]
        item_seq = self.gather_indexes(item_seq_full, seq_length - 1)
        txt_seq = self.gather_indexes(txt_seq_full, seq_length - 1)
        img_seq = self.gather_indexes(img_seq_full, seq_length - 1)
        # 预测
        item_emb_full = self.item_embedding.weight
        txt_emb_full = self.txt_projection(self.txt_embedding.weight)
        img_emb_full = self.img_projection(self.img_embedding.weight)

        item_score = torch.matmul(item_seq, item_emb_full.transpose(0, 1))
        txt_score = torch.matmul(txt_seq, txt_emb_full.transpose(0, 1))
        img_score = torch.matmul(img_seq, img_emb_full.transpose(0, 1))
        score = item_score + txt_score + img_score
        return [item_emb, txt_emb, img_emb], [item_seq, txt_seq, img_seq], score

    def calculate_loss(self, interaction):
        item_idx = interaction[self.ITEM_SEQ]
        item_seq_len = interaction[self.ITEM_SEQ_LEN]
        timestamp = interaction['timestamp_list']
        item_emb_seq, seq_vectors, score = self.forward(item_idx, item_seq_len, timestamp)
        pos_items = interaction[self.POS_ITEM_ID]
        loss = self.loss_fct(score, pos_items)
        return loss + self.IDCL(seq_vectors[0], interaction) + self.CP(item_idx) + self.PCL(interaction, item_emb_seq, seq_vectors)

    def predict(self, interaction):
        item_seq = interaction[self.ITEM_SEQ]
        item_seq_len = interaction[self.ITEM_SEQ_LEN]
        test_item = interaction[self.ITEM_ID]
        timestamp = interaction['timestamp_list']
        _, _, scores = self.forward(item_seq, item_seq_len, timestamp)
        return scores[:, test_item]

    def full_sort_predict(self, interaction):
        item_seq = interaction[self.ITEM_SEQ]
        item_seq_len = interaction[self.ITEM_SEQ_LEN]
        timestamp = interaction['timestamp_list']
        _, _, score = self.forward(item_seq, item_seq_len, timestamp)
        return score

    def IDCL(self, seq_pre, interaction):
        # from UniSRec
        seq_output = F.normalize(seq_pre, dim=1)
        pos_id = interaction['item_id']
        same_pos_id = (pos_id.unsqueeze(1) == pos_id.unsqueeze(0))
        same_pos_id = torch.logical_xor(same_pos_id, torch.eye(pos_id.shape[0], dtype=torch.bool, device=pos_id.device))
        pos_items_emb = self.item_embedding(pos_id)
        pos_items_emb = F.normalize(pos_items_emb, dim=1)

        pos_logits = (seq_output * pos_items_emb).sum(dim=1) / self.temperature
        pos_logits = torch.exp(pos_logits)

        neg_logits = torch.matmul(seq_output, pos_items_emb.transpose(0, 1)) / self.temperature
        neg_logits = torch.where(same_pos_id, torch.tensor([0], dtype=torch.float, device=same_pos_id.device), neg_logits)
        neg_logits = torch.exp(neg_logits).sum(dim=1)

        loss = -torch.log(pos_logits / neg_logits)
        return loss.mean()

    def CP(self, input_idx, padding_idx=0):
        item_list = input_idx.flatten()
        nonzero_idx = torch.where(input_idx != padding_idx) #
        # 嵌入映射
        item_emb = self.item_embedding(item_list)
        txt_emb = self.txt_projection(self.txt_embedding(item_list))
        img_emb = self.img_projection(self.img_embedding(item_list))
        # 预测类别
        item_attribute_score = self.cat_linear(torch.cat([item_emb, txt_emb, img_emb], dim=-1))
        # 获取答案类别
        item_attribute_target = self.cat_embedding(item_list)
        # 计算损失
        attr_loss = self.cat_criterion(item_attribute_score[nonzero_idx], item_attribute_target[nonzero_idx])
        return attr_loss

    def seq2seq_contrastive(self, seq_1, seq_2, same_pos_id):
        seq_1 = F.normalize(seq_1, dim=1)
        seq_2 = F.normalize(seq_2, dim=1)

        pos_logits = (seq_1 * seq_2).sum(dim=1) / self.phcl_temperature
        pos_logits = torch.exp(pos_logits)
        neg_logits = torch.matmul(seq_1, seq_2.transpose(0, 1)) / self.phcl_temperature
        neg_logits = torch.where(same_pos_id, torch.tensor([0], dtype=torch.float, device=same_pos_id.device),neg_logits)
        neg_logits = torch.exp(neg_logits).sum(dim=1)

        loss = -torch.log(pos_logits / neg_logits)
        return loss.mean() * self.phcl_weight

    def PCL(self, interaction, item_emb_seq, seq_embs):
        beta = self.beta
        item_seq = interaction[self.ITEM_SEQ]
        item_seq_len = interaction[self.ITEM_SEQ_LEN]
        timestamp = interaction['timestamp_list']
        # 增强部分
        num_mask = torch.floor(item_seq_len * beta).long().tolist()
        masked_item_seq = item_seq.cpu().detach().numpy().copy()
        for i in range(item_seq.shape[0]):
            mask_index = random.sample(range(item_seq_len[i]), k=num_mask[i])
            masked_item_seq[i, mask_index] = -1
        item_seq_aug = torch.tensor(masked_item_seq, dtype=torch.long, device=item_seq.device)
        # 占位符替换物品
        id_embs, txt_embs, img_embs = item_emb_seq[0], item_emb_seq[1], item_emb_seq[2]
        time_embedding = self.time_moe.get_time_embedding(timestamp)
        placeholder_mask = (item_seq_aug == -1).unsqueeze(2)
        txt_embs_aug = txt_embs.masked_fill(placeholder_mask, 0.0)
        txt_placeholder = self.placeholder_txt(time_embedding).masked_fill(~placeholder_mask, 0.0)
        txt_embs_aug += txt_placeholder
        img_embs_aug = img_embs.masked_fill(placeholder_mask, 0.0)
        img_placeholder = self.placeholder_img(time_embedding).masked_fill(~placeholder_mask, 0.0)
        img_embs_aug += img_placeholder
        # 增强表征计算
        txt_embs_aug = self.dropout(self.txt_ln(txt_embs_aug))
        img_embs_aug = self.dropout(self.img_ln(img_embs_aug))
        extended_attention_mask = self.get_attention_mask(item_seq)
        txt_seq_full = self.txt_seq(txt_embs_aug, extended_attention_mask, output_all_encoded_layers=True)[-1]
        img_seq_full = self.img_seq(img_embs_aug, extended_attention_mask, output_all_encoded_layers=True)[-1]
        txt_seq = self.gather_indexes(txt_seq_full, item_seq_len - 1)
        img_seq = self.gather_indexes(img_seq_full, item_seq_len - 1)
        # 对比学习计算
        pos_id = interaction['item_id']
        same_pos_id = (pos_id.unsqueeze(1) == pos_id.unsqueeze(0))
        same_pos_id = torch.logical_xor(same_pos_id, torch.eye(item_seq.shape[0], dtype=torch.bool, device=item_seq.device))
        txt_loss, img_loss = self.seq2seq_contrastive(seq_embs[1], txt_seq, same_pos_id), self.seq2seq_contrastive(seq_embs[2], img_seq, same_pos_id)
        return (txt_loss + img_loss) / 2


class Align_MoE(nn.Module):
    def __init__(self, config):
        super(Align_MoE, self).__init__()
        self.expert_num = config["start_expert_num"]
        self.hidden_size = int(config["hidden_size"])
        self.gate_selection = config["start_gate_selection"]
        self.gate_txt = nn.Linear(self.hidden_size, self.expert_num)
        self.gate_img = nn.Linear(self.hidden_size, self.expert_num)
        self.gate_id = nn.Linear(self.hidden_size, self.expert_num)
        self.expert = nn.ModuleList([nn.Linear(self.hidden_size * 3, self.hidden_size * 3) for _ in range(self.expert_num)])  # 先实现最简单的专家网络
        self.weight = nn.Parameter(torch.tensor(config["initializer_weight"]).to('cuda'), requires_grad=True)

    def forward(self, vector):
        # 先只实现softmax
        output = None
        if self.gate_selection == 'softmax':
            expert_output = []
            for i in range(self.expert_num):
                expert_output.append(self.expert[i](vector).unsqueeze(2))
            expert_output = torch.cat(expert_output, dim=2)
            output = []
            output.append(self.weight[0] * torch.sum(expert_output[:,:,:,:self.hidden_size] * F.softmax(self.gate_id(vector[:,:,:self.hidden_size]), dim=-1).unsqueeze(3), dim=2))
            output.append(self.weight[1] * torch.sum(expert_output[:,:,:, self.hidden_size:2 * self.hidden_size] * F.softmax(self.gate_txt(vector[:,:,self.hidden_size:2 * self.hidden_size]), dim=-1).unsqueeze(3), dim=2))
            output.append(self.weight[2] * torch.sum(expert_output[:,:,:,2 * self.hidden_size:] * F.softmax(self.gate_img(vector[:,:,2 * self.hidden_size:]), dim=-1).unsqueeze(3), dim=2))
        return output


class Temporal_MoE_C(nn.Module):
    def __init__(self, config):
        super(Temporal_MoE_C, self).__init__()
        self.data_name = config["dataset"].split('/')[-1]
        self.interval_scale = config["interval_scale"]
        self.hidden_size = int(config["hidden_size"])
        self.expert_num = config["temporal_expert_num"]
        self.gate_selection = config["temporal_gate_selection"]
        self.gate = nn.Linear(2 * self.hidden_size, self.expert_num)
        self.absolute_w = nn.Linear(1, self.hidden_size)
        self.absolute_m = nn.Linear(self.hidden_size, self.hidden_size)
        self.time_embedding = nn.Embedding(int(self.interval_scale * self.get_interval_num()) + 1, self.hidden_size)

        self.expert = [nn.Parameter(torch.Tensor(1, self.hidden_size * 3).to('cuda'), requires_grad=True) for _ in range(self.expert_num)]
        for i in range(self.expert_num):
            nn.init.normal_(self.expert[i], std=0.1)

    def get_interval_num(self):
        with open(f'./dataset/{self.data_name}/interval_num', 'rb') as f: return pickle.load(f)

    def get_minmax_day(self):
        with open(f'./dataset/{self.data_name}/minmax_num', 'rb') as f: return pickle.load(f)

    def get_time_embedding(self, timestamp):
        absolute_embedding = torch.cos(self.freq_enhance_ab(self.absolute_w(timestamp.unsqueeze(2))))
        interval_first = torch.zeros((timestamp.shape[0], 1)).long().to('cuda')
        interval = torch.log2(timestamp[:, 1:] - timestamp[:, :-1] + 1)
        interval_index = torch.floor(self.interval_scale * interval).long()
        interval_index = torch.cat([interval_first, interval_index], dim=-1)
        interval_embedding = self.time_embedding(interval_index)
        return torch.cat([interval_embedding, absolute_embedding], dim=-1)

    def freq_enhance_ab(self, timestamp):
        freq = 10000
        freq_seq = torch.arange(0, self.hidden_size, 1.0, dtype=torch.float).to('cuda')
        inv_freq = 1 / torch.pow(freq, (freq_seq / self.hidden_size)).view(1, -1) # shape = (64)
        return timestamp * inv_freq

    def forward(self, vector, timestamp):
        # 先只实现softmax
        expert_proba = None
        absolute_embedding = torch.cos(self.freq_enhance_ab(self.absolute_w(timestamp.unsqueeze(2))))
        interval_first = torch.zeros((vector.shape[0], 1)).long().to('cuda')
        interval = torch.log2(timestamp[:, 1:] - timestamp[:, :-1] + 1)
        interval_index = torch.floor(self.interval_scale * interval).long()
        interval_index = torch.cat([interval_first, interval_index], dim=-1)
        interval_embedding = self.time_embedding(interval_index)
        route = F.softmax(self.gate(torch.cat([interval_embedding, absolute_embedding], dim=-1)), dim=-1)
        if self.gate_selection == 'softmax':
            expert_output = []
            for i in range(self.expert_num):
                expert_output.append((vector * self.expert[i]).unsqueeze(2))
            expert_output = torch.cat(expert_output, dim=2)
            expert_proba = torch.sum(expert_output * route.unsqueeze(3), dim=2)
        return expert_proba[:, :, :self.hidden_size], expert_proba[:, :, self.hidden_size: 2 * self.hidden_size], expert_proba[:, :, 2 * self.hidden_size:]


## 3) Train/eval runner (not dependent on local `run_hm4sr.py`)


In [ ]:
from logging import getLogger
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.data.transform import construct_transform
from recbole.utils import init_logger, get_trainer, init_seed, set_color, get_flops

def print_result(test_result, logger, k: int = 4):
    count = 0
    info = "\ntest result:"
    for metric_name in test_result.keys():
        if count == 0:
            info += "\n"
        count = (count + 1) % k
        info += "{:15}:{:<10}    ".format(metric_name, test_result[metric_name])
    logger.info(info)

def run_hm4sr_notebook(dataset: str = "Games", saved: bool = True):
    config_files = ["./config/data.yaml", f"./config/{dataset}.yaml"]
    config = Config(model=HM4SR, dataset=dataset, config_file_list=config_files)

    init_seed(config["seed"], config["reproducibility"])
    init_logger(config)
    logger = getLogger()
    logger.info(config)

    dataset_obj = create_dataset(config)
    logger.info(dataset_obj)
    train_data, valid_data, test_data = data_preparation(config, dataset_obj)

    init_seed(config["seed"] + config["local_rank"], config["reproducibility"])
    model = HM4SR(config, train_data._dataset).to(config["device"])
    logger.info(model)

    transform = construct_transform(config)
    flops = get_flops(model, dataset_obj, config["device"], logger, transform)
    logger.info(set_color("FLOPs", "blue") + f": {flops}")

    trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)
    best_valid_score, best_valid_result = trainer.fit(
        train_data, valid_data, saved=saved, show_progress=config["show_progress"]
    )
    test_result = trainer.evaluate(
        test_data, load_best_model=saved, show_progress=config["show_progress"]
    )

    logger.info(set_color("best valid ", "yellow") + f": {best_valid_result}")
    logger.info(set_color("test result", "yellow") + f": {test_result}")
    print_result(test_result, logger, k=4)
    return {
        "best_valid_score": best_valid_score,
        "valid_metric_bigger": config["valid_metric_bigger"],
        "best_valid_result": best_valid_result,
        "test_result": test_result,
    }


## 4) Inline `dataprocess` modules


In [ ]:
# Source: dataprocess/get_df.py
import gzip
import pandas as pd
import json

def parse(path):
    g = gzip.open(path, 'rb')
    for l in g:
        yield eval(l)

def get_df(path):
    i = 0
    df = {}
    for d in parse(path):
        df[i] = d
        i += 1
    return pd.DataFrame.from_dict(df, orient='index')

def parse_2018(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield json.loads(l)

def get_df_2018(path):
  i = 0
  df = {}
  for d in parse_2018(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

# Source: dataprocess/args.py
import argparse

def getArgs():
    parser = argparse.ArgumentParser()
    ### 基本参数
    parser.add_argument('--dataset', type=str, default='Games')
    parser.add_argument('--batch_size', type=int, default=1024)
    parser.add_argument('--max_length', type=int, default=50)
    parser.add_argument('--max_epoch', type=int, default=1000)
    parser.add_argument('--random_seed', type=int, default=2024)
    parser.add_argument('--lr', type=float, default=1e-3)
    parser.add_argument('--device', type=str, default='cuda')
    parser.add_argument('--tolerance', type=int, default=10)
    ### 模型参数
    parser.add_argument('--hidden_dim', type=int, default=64)
    parser.add_argument('--seq_head', type=int, default=2)
    parser.add_argument('--seq_layers', type=int, default=2)
    parser.add_argument('--dropout', type=float, default=0.5)

    args = parser.parse_args()
    ### 路径参数
    args.txt_emb = f'./dataset/{args.dataset}/txt_emb.pt'
    args.img_emb = f'./dataset/{args.dataset}/img_emb.pt'
    args.ckpt = f'./ckpt/{args.dataset}'
    args.item_dict_path = f'./dataset/{args.dataset}/item2id'
    args.log_path = f'./log/{args.dataset}_{args.random_seed}.txt'
    args.inter_path = f'./dataset/{args.dataset}/{args.dataset}.inter'
    args.data_path = f'./dataset/{args.dataset}/00_seq'
    args.txt_path = f'./dataset/{args.dataset}/content.txt'
    args.img_path = f'./dataset/{args.dataset}/image/'
    args.meta_path = f'./dataset/meta_{args.dataset}.json.gz'
    return args

# Source: dataprocess/txt_extractor.py
import torch
import html
import re
import numpy as np
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer
from pytorch_transformers import BertModel, BertConfig, BertTokenizer
from get_df import get_df

def clean_text(raw_text):
    # from UniSRec
    if isinstance(raw_text, list):
        cleaned_text = ' '.join(raw_text[0])
    elif isinstance(raw_text, dict):
        cleaned_text = str(raw_text)
    else:
        cleaned_text = raw_text
    cleaned_text = html.unescape(cleaned_text)
    cleaned_text = re.sub(r'["\n\r]*', '', cleaned_text)
    cleaned_text = cleaned_text.replace('\t', ' ')
    index = -1
    while -index < len(cleaned_text) and cleaned_text[index] == '.':
        index -= 1
    index += 1
    if index == 0:
        cleaned_text = cleaned_text + '.'
    else:
        cleaned_text = cleaned_text[:index] + '.'
    if len(cleaned_text) >= 2000:
        cleaned_text = ''
    return cleaned_text

def generate_text(args, items, features):
    def not_nan(nan):
        return nan == nan
    item_text_list = []
    already_items = set()
    data_df = get_df(args.meta_path)

    for index in tqdm(range(data_df.shape[0]), desc='Generate text'):
        item = data_df['asin'][index]
        if item in items and item not in already_items:
            already_items.add(item)
            text = ''
            for meta_key in features:
                if meta_key in data_df.columns:
                    content = data_df[meta_key][index]
                    if not_nan(content):
                        meta_value = clean_text(content)
                        text += meta_value + ' '
            item_text_list.append((item, text))
    with open(args.txt_path, 'w') as f:
        for i, record in enumerate(item_text_list):
            line = '\t'.join(record)
            f.write(line + '\n')
    return item_text_list

def load_content(args):
    item_text_list = []
    with open(args.txt_path, 'r') as f:
        while True:
            line = f.readline().strip()
            if len(line) == 0: break
            item, text = line.split('\t')
            item_text_list.append((item, text))
    return item_text_list

def txt_extractor(sentences, args, padding_idx=0):
    # 要求sentence必须事先按id排好序
    tokenizer = BertTokenizer.from_pretrained('./pretrained/bert-base-uncased/vocab.txt')
    config = BertConfig.from_pretrained('./pretrained/bert-base-uncased/config.json')
    bert = BertModel.from_pretrained('./pretrained/bert-base-uncased/pytorch_model.bin', config=config).to(args.device)
    result = []
    with torch.no_grad():
        for _, sentence in tqdm(enumerate(sentences), desc='Text Extracting', total=len(sentences)):
            sentence = '[CLS] ' + sentence
            token_seq = tokenizer.tokenize(sentence)
            idx_seq = tokenizer.convert_tokens_to_ids(token_seq)
            idx_seq_tensor = torch.tensor(idx_seq, dtype=torch.long).to(args.device).view(1, -1)
            output = bert(idx_seq_tensor)
            result.append(output[0][:, 0, :].cpu())
        result.insert(padding_idx, torch.zeros(result[-1].shape, dtype=torch.float))
        txt_emb = torch.cat(result, dim=0)
    torch.save(txt_emb, args.txt_emb)


# Source: dataprocess/img_extractor.py
import torch
import os
from tqdm import tqdm
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from timm.models.vision_transformer import vit_base_patch16_clip_224

class ImgDataset(Dataset):
    def __init__(self, images_path: list, images_class: list, transform=None):
        self.images_path = images_path
        self.images_class = images_class
        self.transform = transform

    def __len__(self):
        return len(self.images_path)

    def __getitem__(self, item):
        img = Image.open(self.images_path[item])
        label = self.images_class[item]
        if self.transform is not None:
            img = self.transform(img)
        return img, label

    @staticmethod
    def collate_fn(batch):
        images, labels = tuple(zip(*batch))
        images = torch.stack(images, dim=0)
        labels = torch.as_tensor(labels)
        return images, labels

class ImgExtractTool:
    def __init__(self,
                 args,
                 model_path='./pretrained/vit_base_patch16_clip_224.pth'):
        self.device = args.device
        self.model_path = model_path
        self.basic_path = f'./dataset/{args.dataset}/image/'
        self.feature_path = f'./dataset/{args.dataset}/feature/'
        self.model = self.load_weight()
        self.data_transform = {
            'val': transforms.Compose([transforms.Resize(256),
                                       transforms.CenterCrop(224),
                                       transforms.ToTensor(),
                                       transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])}

    def load_weight(self):
        model = vit_base_patch16_clip_224()
        weights_dict = torch.load(self.model_path)
        print(model.load_state_dict(weights_dict, strict=False))
        return model.to(self.device)

    @staticmethod
    def replace_RGB(file_path):
        img = Image.open(file_path)
        if img.mode != 'RGB':
            # print("image: {} isn't RGB mode.".format(file_path))
            img_rgb = img.convert("RGB")
            os.remove(file_path)
            img_rgb.save(file_path)

    def extract_one_instance(self, instance):
        # 这里每个物品有且只有一个图片，所以也不用考虑batch的问题
        target_path = self.basic_path + instance + '/'
        image_name = os.listdir(target_path)[0]
        image_path = os.path.join(target_path, image_name)
        try:
            self.replace_RGB(image_path)
        except:
            return torch.zeros((1, 768), dtype=torch.float)

        dataset = ImgDataset([image_path], [0], transform=self.data_transform['val'])
        assert len(dataset) <= 1
        dataloader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=dataset.collate_fn)
        for _, data in enumerate(dataloader): # 实际上只会有一个batch
            img, _ = data
            output = self.model.forward_features(img.to(self.device)).cpu()
            feature = output[:, 0].view(1, 768)
            return feature

        return torch.zeros((1, 768), dtype=torch.float)

def img_extractor(args, item_id_list, padding_idx=0):
    # 要求item_id_list必须事先按id排好序
    with torch.no_grad():
        tool = ImgExtractTool(args)
        result = []
        for _, t in tqdm(enumerate(item_id_list), desc='Image Extracting', total=len(item_id_list)):
            result.append(tool.extract_one_instance(t))
        result.insert(padding_idx, torch.zeros((1, 768), dtype=torch.float))
        img_emb = torch.cat(result, dim=0)
    torch.save(img_emb, args.img_emb)



# Source: dataprocess/data_process.py
import os
import pickle
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
# inlined: generate_text, txt_extractor, load_content
# inlined: img_extractor
# inlined: get_df
from tqdm import tqdm

def amazon(args):
    data_path = f'./dataset/reviews_{args.dataset}_5.json.gz'
    data_df = get_df(data_path)
    inter_df = data_df.rename(
        columns={'reviewerID': 'user', 'asin': 'item', 'unixReviewTime': 'timestamp', 'overall': 'stars'})
    inter_df = inter_df[['user', 'item', 'timestamp']]
    user_list = sorted(inter_df['user'].unique())
    user2id = dict(zip(user_list, range(1, len(user_list) + 1)))
    inter_df['user'] = inter_df['user'].apply(lambda x: user2id[x])
    return inter_df

def inter2txt(inter_df, txt_path):
    df = inter_df.sort_values(by=['user', 'timestamp'], kind='mergesort').reset_index(drop=True)
    with open(txt_path, 'w') as f:
        f.write('user_id:token\titem_id:token\ttimestamp:float\n')
        for i, row in tqdm(df.iterrows(), desc='Generating inter file', total=df.shape[0]):
            user, item, t = row['user'], row['item'], row['timestamp']
            f.write('{}\t{}\t{}\n'.format(user, item, t))

def recbole2local(config, dataloader, local_path):
    uid_list, seq, target, interval, length = [], [], [], [], []
    user_field = config["USER_ID_FIELD"]
    seq_field = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
    target_field = config["ITEM_ID_FIELD"]
    length_field = config["ITEM_LIST_LENGTH_FIELD"]
    interval_field = config["TIME_FIELD"] + config["LIST_SUFFIX"]
    for _, interaction in enumerate(dataloader):
        uid_list.append(interaction[user_field].long())
        seq.append(interaction[seq_field].long())
        target.append(interaction[target_field].long())
        interval.append(interaction[interval_field].long())
        length.append(interaction[length_field].long())
    uid_list, seq, target, interval, length = torch.cat(uid_list, dim=0), torch.cat(seq, dim=0), torch.cat(target, dim=0), torch.cat(interval, dim=0), torch.cat(length, dim=0)
    with open(local_path, 'wb') as f: pickle.dump((uid_list, seq, target, interval, length), f)

# the dataloaders of training and testing from recbole are unexpectedly different
def recbole2local_val(config, dataloader, local_path):
    uid_list, seq, target, interval, length = [], [], [], [], []
    user_field = config["USER_ID_FIELD"]
    seq_field = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
    target_field = config["ITEM_ID_FIELD"]
    length_field = config["ITEM_LIST_LENGTH_FIELD"]
    interval_field = config["TIME_FIELD"] + config["LIST_SUFFIX"]
    for _, inter in enumerate(dataloader):
        interaction = inter[0]
        uid_list.append(interaction[user_field].long())
        seq.append(interaction[seq_field].long())
        target.append(interaction[target_field].long())
        interval.append(interaction[interval_field].long())
        length.append(interaction[length_field].long())
        length.append(interaction[length_field].long())
    uid_list, seq, target, interval, length = torch.cat(uid_list, dim=0), torch.cat(seq, dim=0), torch.cat(target, dim=0), torch.cat(interval, dim=0), torch.cat(length, dim=0)
    with open(local_path, 'wb') as f: pickle.dump((uid_list, seq, target, interval, length), f)

def prepare_inter(args):
    if not os.path.exists(args.inter_path):
        inter_df = amazon(args)
        inter2txt(inter_df, args.inter_path)

def prepare_txt_emb(args):
    if not os.path.exists(args.txt_emb):
        with open(args.item_dict_path, 'rb') as f: item2id = pickle.load(f)
        if not os.path.exists(args.txt_path):
            item_sentences = generate_text(args, item2id.keys(), ['title', 'categories', 'brand']) # omit category for future use
        else: item_sentences = load_content(args)
        item_sentences = sorted(item_sentences, key=lambda x: item2id[x[0]])
        sentences = [s[1] for s in item_sentences]
        txt_extractor(sentences, args)

def prepare_img_emb(args):
    with open(args.item_dict_path, 'rb') as f: item2id = pickle.load(f)
    item2id.pop('[PAD]')
    if not os.path.exists(args.img_emb):
        item_id_list = list(item2id.keys())
        item_id_list = sorted(item_id_list, key=lambda x: item2id[x])
        img_extractor(args, item_id_list)

def local_timestamp(args, data_path):
    with open(args.data_path.replace('00_seq', args.dataset + '.inter'), 'r') as f:
        line = f.readlines()[1:]
    tmp = [-1, 0]
    max_interval = 0
    for l in line:
        user, _, timestamp = l.strip().split('\t')
        user, timestamp = int(user), int(timestamp)
        if user != tmp[0]:
            tmp = [user, timestamp]
        else:
            interval = timestamp - tmp[1]
            tmp[1] = timestamp
            max_interval = max(interval, max_interval)
    max_interval = torch.log2(torch.tensor(max_interval) + 1).item()
    with open(data_path, 'wb') as f:
        print(max_interval)
        pickle.dump(max_interval, f)

def local_minmax_day(args, data_path):
    with open(args.data_path.replace('00_seq', args.dataset + '.inter'), 'r') as f:
        line = f.readlines()[1:]
    min_date, max_date = 9999999, 0
    for l in line:
        user, _, timestamp = l.strip().split('\t')
        user, timestamp = int(user), int(timestamp)
        min_date = min(int(timestamp / 86400), min_date)
        max_date = max(int(timestamp / 86400), max_date)
    with open(data_path, 'wb') as f:
        print(min_date)
        print(max_date)
        pickle.dump((min_date, max_date), f)

def prepare_seq(args):
    if not os.path.exists(args.data_path.replace('00', 'train')):
        config = Config(model='SASRec', dataset=f'./dataset/{args.dataset}/{args.dataset}', config_file_list=['./config/data.yaml'])
        dataset = create_dataset(config)
        train_data, dev_data, test_data = data_preparation(config, dataset)
        # recbole对物品进行了重新映射，因此需要将新的映射改写到原来的item2id中
        item2id = train_data.dataset.field2token_id['item_id']
        with open(args.item_dict_path, 'wb') as f: pickle.dump(item2id, f)
        train_data.shuffle = False
        recbole2local(config, train_data, args.data_path.replace('00', 'train'))
        recbole2local_val(config, dev_data, args.data_path.replace('00', 'dev'))
        recbole2local_val(config, test_data, args.data_path.replace('00', 'test'))
    if not os.path.exists(args.data_path.replace('00_seq', 'interval_num')):
        local_timestamp(args, args.data_path.replace('00_seq', 'interval_num'))
    if not os.path.exists(args.data_path.replace('00_seq', 'minmax_day')):
        local_minmax_day(args, args.data_path.replace('00_seq', 'minmax_num'))

def prepare_category(args, padding_idx=0):
    if not os.path.exists(args.data_path.replace('00_seq', 'cat.pt')):
        with open(args.item_dict_path, 'rb') as f: item2id = pickle.load(f)
        data_df = get_df(args.meta_path)
        cat_dict = {}
        cat_type_dict = {}
        for i in range(data_df.shape[0]):
            item = data_df['asin'][i]
            if item in item2id.keys():
                item_id = item2id[item]
                category = data_df['categories'][i][0]
                category_id = []
                for c in category:
                    if c not in cat_type_dict.keys():
                        cat_type_dict[c] = len(cat_type_dict)
                    category_id.append(cat_type_dict[c])
                category_id = torch.tensor(category_id)
                cat_dict[item_id] = category_id
        result = []
        for i in range(1, len(item2id)):
            ht = torch.nn.functional.one_hot(cat_dict[i], num_classes=len(cat_type_dict)).sum(dim=0).view(1, -1)
            result.append(ht)
        result.insert(padding_idx, torch.zeros((1, len(cat_type_dict)), dtype=torch.long))
        result = torch.cat(result, dim=0)
        torch.save(result, args.data_path.replace('00_seq', 'cat.pt'))


## 5) Optional preprocessing run

Uncomment if your Kaggle input does not already contain processed files.


In [ ]:
# OPTIONAL preprocess
# args = getArgs()
# prepare_inter(args)
# prepare_seq(args)
# prepare_txt_emb(args)
# prepare_img_emb(args)
# prepare_category(args)
# print("Preprocess done")


## 6) Train + evaluate


In [ ]:
result = run_hm4sr_notebook(dataset="Games", saved=True)
result
